In [1]:
%pip install nltk
import sys
!{sys.executable} -m pip install tensorflow
import tensorflow as tf

print("TensorFlow Version:", tf.__version__)
print("GPU Available:", tf.config.list_physical_devices('GPU'))

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, LSTM, Dense

print("All TensorFlow Keras imports successful!")

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


  Using cached tensorflow-2.21.0-cp313-cp313-win_amd64.whl.metadata (4.5 kB)
  Using cached absl_py-2.5.1-py3-none-any.whl.metadata (3.3 kB)
  Using cached astunparse-1.6.3-py2.py3-none-any.whl.metadata (4.4 kB)
  Using cached flatbuffers-25.12.19-py2.py3-none-any.whl.metadata (1.0 kB)
  Using cached gast-0.7.0-py3-none-any.whl.metadata (1.5 kB)
  Using cached google_pasta-0.2.0-py3-none-any.whl.metadata (814 bytes)
  Using cached libclang-18.1.1-py2.py3-none-win_amd64.whl.metadata (5.3 kB)
  Using cached opt_einsum-3.4.0-py3-none-any.whl.metadata (6.3 kB)
  Using cached protobuf-7.36.2-cp310-abi3-win_amd64.whl.metadata (595 bytes)
  Using cached setuptools-84.2.1-py3-none-any.whl.metadata (6.6 kB)
  Using cached termcolor-3.3.0-py3-none-any.whl.metadata (6.5 kB)
  Using cached wrapt-2.5.2-cp313-cp313-win_amd64.whl.metadata (7.7 kB)
  Using cached keras-3.15.1-py3-none-any.whl.metadata (6.4 kB)
  Using cached h5py-3.14.0-cp313-cp313-win_amd64.whl.metadata (2.7 kB)
  Using cached ml_dty

In [2]:
import nltk
print("NLTK installed successfully!")

NLTK installed successfully!


In [3]:
import pandas as pd
import numpy as np
import re
import nltk
import joblib
import pickle

# Scikit-learn
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

# TensorFlow
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

from tensorflow.keras.models import Sequential

from tensorflow.keras.layers import (
    Embedding,
    SimpleRNN,
    LSTM,
    Dense,
    Dropout
)

from tensorflow.keras.callbacks import EarlyStopping


# ============================================================
# 1. DOWNLOAD NLTK DATA
# ============================================================

nltk.download("stopwords")
nltk.download("wordnet")
nltk.download("omw-1.4")

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer


# ============================================================
# 2. LOAD DATA
# ============================================================

fake = pd.read_csv(
    r"C:\Users\Ezhilarasi\OneDrive\Desktop\AI_ML_Project\NLP\Fake news prediction\Fake.csv"
)

real = pd.read_csv(
    r"C:\Users\Ezhilarasi\OneDrive\Desktop\AI_ML_Project\NLP\Fake news prediction\True.csv"
)


# ============================================================
# 3. ADD LABEL
# ============================================================

# Fake = 0
# Real = 1

fake["label"] = 0
real["label"] = 1


# ============================================================
# 4. COMBINE DATA
# ============================================================

df = pd.concat(
    [fake, real],
    ignore_index=True
)


print("=" * 60)
print("DATASET INFORMATION")
print("=" * 60)

print("Fake data:", fake.shape)
print("Real data:", real.shape)
print("Combined data:", df.shape)


# ============================================================
# 5. HANDLE MISSING VALUES
# ============================================================

df["title"] = df["title"].fillna("")
df["text"] = df["text"].fillna("")


# ============================================================
# 6. COMBINE TITLE + TEXT
# ============================================================

df["content"] = (
    df["title"] + " " + df["text"]
)


# ============================================================
# 7. TEXT CLEANING
# ============================================================

stop_words = set(
    stopwords.words("english")
)

lemmatizer = WordNetLemmatizer()


def clean_text(text):

    text = str(text)

    # Lowercase
    text = text.lower()

    # Remove HTML
    text = re.sub(
        r"<.*?>",
        " ",
        text
    )

    # Remove URLs
    text = re.sub(
        r"http\S+|www\S+",
        " ",
        text
    )

    # Remove email
    text = re.sub(
        r"\S+@\S+",
        " ",
        text
    )

    # Keep English alphabets
    text = re.sub(
        r"[^a-zA-Z\s]",
        " ",
        text
    )

    # Remove extra spaces
    text = re.sub(
        r"\s+",
        " ",
        text
    ).strip()

    # Tokenization
    tokens = text.split()

    # Stopword removal
    tokens = [
        word
        for word in tokens
        if word not in stop_words
    ]

    # Lemmatization
    tokens = [
        lemmatizer.lemmatize(word)
        for word in tokens
    ]

    return " ".join(tokens)


print("\nCleaning text...")

df["processed_text"] = (
    df["content"].apply(clean_text)
)


# ============================================================
# 8. REMOVE DUPLICATES
# ============================================================

before = len(df)

df = df.drop_duplicates(
    subset=["processed_text"]
)

after = len(df)

print("Duplicates removed:", before - after)


# ============================================================
# 9. INPUT AND TARGET
# ============================================================

X = df["processed_text"]
y = df["label"]


# ============================================================
# 10. TRAIN TEST SPLIT
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(

    X,
    y,

    test_size=0.20,

    random_state=42,

    stratify=y
)


print("\nTraining samples:", len(X_train))
print("Testing samples :", len(X_test))


# ============================================================
# ============================================================
# MODEL 1 - LOGISTIC REGRESSION
# ============================================================
# ============================================================

print("\n")
print("=" * 60)
print("MODEL 1: LOGISTIC REGRESSION")
print("=" * 60)


# ============================================================
# TF-IDF
# ============================================================

tfidf = TfidfVectorizer(

    ngram_range=(1, 2),

    max_features=50000,

    min_df=2,

    max_df=0.95
)


X_train_tfidf = tfidf.fit_transform(
    X_train
)

X_test_tfidf = tfidf.transform(
    X_test
)


print(
    "TF-IDF training shape:",
    X_train_tfidf.shape
)


# ============================================================
# LOGISTIC REGRESSION
# ============================================================

logistic_model = LogisticRegression(
    max_iter=1000
)


print("Training Logistic Regression...")

logistic_model.fit(
    X_train_tfidf,
    y_train
)

print("Training completed!")


# ============================================================
# LOGISTIC PREDICTION
# ============================================================

logistic_pred = logistic_model.predict(
    X_test_tfidf
)


# ============================================================
# LOGISTIC EVALUATION
# ============================================================

logistic_accuracy = accuracy_score(
    y_test,
    logistic_pred
)

logistic_precision = precision_score(
    y_test,
    logistic_pred
)

logistic_recall = recall_score(
    y_test,
    logistic_pred
)

logistic_f1 = f1_score(
    y_test,
    logistic_pred
)


print("\nLOGISTIC REGRESSION RESULTS")

print(
    f"Accuracy : {logistic_accuracy:.4f}"
)

print(
    f"Precision: {logistic_precision:.4f}"
)

print(
    f"Recall   : {logistic_recall:.4f}"
)

print(
    f"F1 Score : {logistic_f1:.4f}"
)

print("\nClassification Report")

print(
    classification_report(
        y_test,
        logistic_pred,
        target_names=[
            "Fake",
            "Real"
        ]
    )
)

print("\nConfusion Matrix")

print(
    confusion_matrix(
        y_test,
        logistic_pred
    )
)


# ============================================================
# SAVE LOGISTIC MODEL
# ============================================================

joblib.dump(
    logistic_model,
    "logistic_model.pkl"
)

joblib.dump(
    tfidf,
    "tfidf_vectorizer.pkl"
)

print("\nLogistic model saved!")


# ============================================================
# ============================================================
# TOKENIZATION FOR RNN + LSTM
# ============================================================
# ============================================================

print("\n")
print("=" * 60)
print("TOKENIZATION FOR RNN AND LSTM")
print("=" * 60)


MAX_WORDS = 30000

MAX_LENGTH = 300


tokenizer = Tokenizer(

    num_words=MAX_WORDS,

    oov_token="<OOV>"
)


# IMPORTANT:
# Fit tokenizer only on training data

tokenizer.fit_on_texts(
    X_train
)


# Convert text to sequences

X_train_sequences = (
    tokenizer.texts_to_sequences(
        X_train
    )
)

X_test_sequences = (
    tokenizer.texts_to_sequences(
        X_test
    )
)


# ============================================================
# PADDING
# ============================================================

X_train_padded = pad_sequences(

    X_train_sequences,

    maxlen=MAX_LENGTH,

    padding="post",

    truncating="post"
)


X_test_padded = pad_sequences(

    X_test_sequences,

    maxlen=MAX_LENGTH,

    padding="post",

    truncating="post"
)


print(
    "Training sequence shape:",
    X_train_padded.shape
)

print(
    "Testing sequence shape:",
    X_test_padded.shape
)


# ============================================================
# SAVE TOKENIZER
# ============================================================

with open(
    "tokenizer.pkl",
    "wb"
) as file:

    pickle.dump(
        tokenizer,
        file
    )


# ============================================================
# ============================================================
# MODEL 2 - RNN
# ============================================================
# ============================================================

print("\n")
print("=" * 60)
print("MODEL 2: RNN")
print("=" * 60)


# ============================================================
# BUILD RNN
# ============================================================

rnn_model = Sequential()


rnn_model.add(
    Embedding(

        input_dim=MAX_WORDS,

        output_dim=128,

        input_length=MAX_LENGTH
    )
)


rnn_model.add(
    SimpleRNN(
        128
    )
)


rnn_model.add(
    Dropout(0.5)
)


rnn_model.add(
    Dense(

        64,

        activation="relu"
    )
)


rnn_model.add(
    Dropout(0.5)
)


rnn_model.add(
    Dense(

        1,

        activation="sigmoid"
    )
)


# ============================================================
# COMPILE RNN
# ============================================================

rnn_model.compile(

    optimizer="adam",

    loss="binary_crossentropy",

    metrics=["accuracy"]
)


print("\nRNN Model Summary")

rnn_model.summary()


# ============================================================
# EARLY STOPPING
# ============================================================

early_stop_rnn = EarlyStopping(

    monitor="val_loss",

    patience=3,

    restore_best_weights=True
)


# ============================================================
# TRAIN RNN
# ============================================================

print("\nTraining RNN...")


rnn_history = rnn_model.fit(

    X_train_padded,

    y_train,

    validation_split=0.20,

    epochs=10,

    batch_size=64,

    callbacks=[
        early_stop_rnn
    ]
)


# ============================================================
# RNN PREDICTION
# ============================================================

rnn_probability = (
    rnn_model.predict(
        X_test_padded
    ).ravel()
)


rnn_pred = (
    rnn_probability >= 0.5
).astype(int)


# ============================================================
# RNN EVALUATION
# ============================================================

rnn_accuracy = accuracy_score(
    y_test,
    rnn_pred
)

rnn_precision = precision_score(
    y_test,
    rnn_pred
)

rnn_recall = recall_score(
    y_test,
    rnn_pred
)

rnn_f1 = f1_score(
    y_test,
    rnn_pred
)


print("\nRNN RESULTS")

print(
    f"Accuracy : {rnn_accuracy:.4f}"
)

print(
    f"Precision: {rnn_precision:.4f}"
)

print(
    f"Recall   : {rnn_recall:.4f}"
)

print(
    f"F1 Score : {rnn_f1:.4f}"
)


print("\nClassification Report")

print(
    classification_report(
        y_test,
        rnn_pred,
        target_names=[
            "Fake",
            "Real"
        ]
    )
)


print("\nConfusion Matrix")

print(
    confusion_matrix(
        y_test,
        rnn_pred
    )
)


# ============================================================
# SAVE RNN
# ============================================================

rnn_model.save(
    "fake_news_rnn.keras"
)

print("\nRNN model saved!")


# ============================================================
# ============================================================
# MODEL 3 - LSTM
# ============================================================
# ============================================================

print("\n")
print("=" * 60)
print("MODEL 3: LSTM")
print("=" * 60)


# ============================================================
# BUILD LSTM
# ============================================================

lstm_model = Sequential()


lstm_model.add(
    Embedding(

        input_dim=MAX_WORDS,

        output_dim=128,

        input_length=MAX_LENGTH
    )
)


lstm_model.add(
    LSTM(
        128
    )
)


lstm_model.add(
    Dropout(0.5)
)


lstm_model.add(
    Dense(

        64,

        activation="relu"
    )
)


lstm_model.add(
    Dropout(0.5)
)


lstm_model.add(
    Dense(

        1,

        activation="sigmoid"
    )
)


# ============================================================
# COMPILE LSTM
# ============================================================

lstm_model.compile(

    optimizer="adam",

    loss="binary_crossentropy",

    metrics=["accuracy"]
)


print("\nLSTM Model Summary")

lstm_model.summary()


# ============================================================
# EARLY STOPPING
# ============================================================

early_stop_lstm = EarlyStopping(

    monitor="val_loss",

    patience=3,

    restore_best_weights=True
)


# ============================================================
# TRAIN LSTM
# ============================================================

print("\nTraining LSTM...")


lstm_history = lstm_model.fit(

    X_train_padded,

    y_train,

    validation_split=0.20,

    epochs=10,

    batch_size=64,

    callbacks=[
        early_stop_lstm
    ]
)


# ============================================================
# LSTM PREDICTION
# ============================================================

lstm_probability = (
    lstm_model.predict(
        X_test_padded
    ).ravel()
)


lstm_pred = (
    lstm_probability >= 0.5
).astype(int)


# ============================================================
# LSTM EVALUATION
# ============================================================

lstm_accuracy = accuracy_score(
    y_test,
    lstm_pred
)

lstm_precision = precision_score(
    y_test,
    lstm_pred
)

lstm_recall = recall_score(
    y_test,
    lstm_pred
)

lstm_f1 = f1_score(
    y_test,
    lstm_pred
)


print("\nLSTM RESULTS")

print(
    f"Accuracy : {lstm_accuracy:.4f}"
)

print(
    f"Precision: {lstm_precision:.4f}"
)

print(
    f"Recall   : {lstm_recall:.4f}"
)

print(
    f"F1 Score : {lstm_f1:.4f}"
)


print("\nClassification Report")

print(
    classification_report(
        y_test,
        lstm_pred,
        target_names=[
            "Fake",
            "Real"
        ]
    )
)


print("\nConfusion Matrix")

print(
    confusion_matrix(
        y_test,
        lstm_pred
    )
)


# ============================================================
# SAVE LSTM
# ============================================================

lstm_model.save(
    "fake_news_lstm.keras"
)

print("\nLSTM model saved!")


# ============================================================
# FINAL COMPARISON
# ============================================================

print("\n")
print("=" * 70)
print("FINAL MODEL COMPARISON")
print("=" * 70)


print(
    f"{'Model':<25}"
    f"{'Accuracy':<15}"
    f"{'Precision':<15}"
    f"{'Recall':<15}"
    f"{'F1':<15}"
)

print("-" * 70)


print(
    f"{'Logistic Regression':<25}"
    f"{logistic_accuracy:<15.4f}"
    f"{logistic_precision:<15.4f}"
    f"{logistic_recall:<15.4f}"
    f"{logistic_f1:<15.4f}"
)


print(
    f"{'RNN':<25}"
    f"{rnn_accuracy:<15.4f}"
    f"{rnn_precision:<15.4f}"
    f"{rnn_recall:<15.4f}"
    f"{rnn_f1:<15.4f}"
)


print(
    f"{'LSTM':<25}"
    f"{lstm_accuracy:<15.4f}"
    f"{lstm_precision:<15.4f}"
    f"{lstm_recall:<15.4f}"
    f"{lstm_f1:<15.4f}"
)


print("=" * 70)

print("\nAll three models trained successfully!")

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Ezhilarasi\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Ezhilarasi\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\Ezhilarasi\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


DATASET INFORMATION
Fake data: (23481, 5)
Real data: (21417, 5)
Combined data: (44898, 5)

Cleaning text...
Duplicates removed: 6069

Training samples: 31063
Testing samples : 7766


MODEL 1: LOGISTIC REGRESSION
TF-IDF training shape: (31063, 50000)
Training Logistic Regression...
Training completed!

LOGISTIC REGRESSION RESULTS
Accuracy : 0.9875
Precision: 0.9857
Recall   : 0.9912
F1 Score : 0.9884

Classification Report
              precision    recall  f1-score   support

        Fake       0.99      0.98      0.99      3580
        Real       0.99      0.99      0.99      4186

    accuracy                           0.99      7766
   macro avg       0.99      0.99      0.99      7766
weighted avg       0.99      0.99      0.99      7766


Confusion Matrix
[[3520   60]
 [  37 4149]]

Logistic model saved!


TOKENIZATION FOR RNN AND LSTM
Training sequence shape: (31063, 300)
Testing sequence shape: (7766, 300)


MODEL 2: RNN


c:\Users\Ezhilarasi\OneDrive\Desktop\AI_ML_Project\venv\Lib\site-packages\keras\src\layers\core\embedding.py:123: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(



RNN Model Summary


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

... (7 outputs left out)

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)


Training LSTM...
Epoch 1/10
389/389 ━━━━━━━━━━━━━━━━━━━━ 136s 344ms/step - accuracy: 0.7196 - loss: 0.5466 - val_accuracy: 0.8019 - val_loss: 0.4897
Epoch 2/10
389/389 ━━━━━━━━━━━━━━━━━━━━ 126s 304ms/step - accuracy: 0.8462 - loss: 0.4217 - val_accuracy: 0.8519 - val_loss: 0.3868
Epoch 3/10
389/389 ━━━━━━━━━━━━━━━━━━━━ 118s 302ms/step - accuracy: 0.8272 - loss: 0.4393 - val_accuracy: 0.8339 - val_loss: 0.4392
Epoch 4/10
389/389 ━━━━━━━━━━━━━━━━━━━━ 116s 298ms/step - accuracy: 0.8399 - loss: 0.4156 - val_accuracy: 0.8434 - val_loss: 0.3653
Epoch 5/10
389/389 ━━━━━━━━━━━━━━━━━━━━ 205s 528ms/step - accuracy: 0.9031 - loss: 0.2712 - val_accuracy: 0.9580 - val_loss: 0.1591
Epoch 6/10
389/389 ━━━━━━━━━━━━━━━━━━━━ 248s 637ms/step - accuracy: 0.7018 - loss: 0.5387 - val_accuracy: 0.9604 - val_loss: 0.1307
Epoch 7/10
389/389 ━━━━━━━━━━━━━━━━━━━━ 199s 475ms/step - accuracy: 0.9487 - loss: 0.1683 - val_accuracy: 0.9815 - val_loss: 0.0619
Epoch 8/10
389/389 ━━━━━━━━━━━━━━━━━━━━ 157s 358ms/step - 